In [1]:
from hcipy import (
    make_pupil_grid, make_focal_grid, make_circular_aperture,
    FraunhoferPropagator, make_zernike_basis, Wavefront, Field,
)
from skimage.transform import resize
from image_sharpening import FocusDiversePhaseRetrieval, mft_rev, InstrumentConfiguration


ModuleNotFoundError: No module named 'image_sharpening'

In [ ]:
seal_parameters = {
    'image_dx': 2.0071, 'focal_length_meters': 500e-3, 'wavelength_meter': 650e-9,
    'pupil_size': 10.12e-3, 'pupil_pixel_dimension': 256, 'q': 16, 'Num_airycircles': 16,
}
seal_param_config = {
    'image_dx':   seal_parameters['image_dx'],
    'efl':        seal_parameters['focal_length_meters'] * 1e3,
    'wavelength': seal_parameters['wavelength_meter'] * 1e6,
    'pupil_size': seal_parameters['pupil_size'] * 1e3,
}
WAVELENGTH_UM = seal_param_config['wavelength']
IMAGE_DX_UM   = seal_param_config['image_dx']

conf       = InstrumentConfiguration(seal_param_config)
pupil_grid = make_pupil_grid(seal_parameters['pupil_pixel_dimension'], seal_parameters['pupil_size'])
focal_grid = make_focal_grid(q=seal_parameters['q'], num_airy=seal_parameters['Num_airycircles'],
                             pupil_diameter=seal_parameters['pupil_size'],
                             focal_length=seal_parameters['focal_length_meters'],
                             reference_wavelength=seal_parameters['wavelength_meter'])
aperture        = make_circular_aperture(seal_parameters['pupil_size'])
telescope_pupil = aperture(pupil_grid)
pupil_mask      = np.array(telescope_pupil.shaped, dtype=bool)
prop_p2f        = FraunhoferPropagator(pupil_grid, focal_grid, seal_parameters['focal_length_meters'])
zernike_modes   = make_zernike_basis(num_modes=32, D=seal_parameters['pupil_size'], grid=pupil_grid)
defocus_tmpl    = zernike_modes[3]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Simulation Parameters

We fix a single diagnostic operating point --- $A = 0.10$ waves of sinusoidal phase at
$\nu_0 = 10$ cyc/aperture, retrieved with a three-image diversity stack at
$\Delta z = 40$ mm --- and vary one knob at a time downstream.  The point sits well
inside the regime in which the sideband expansion of Section~\ref{sec:otf_theory}
of the paper is valid ($A \ll 1$ rad, $\nu_0 < \nu_{\rm cutoff}$); the goal is
not to map a parameter space but to characterize what the FDPR pipeline returns at
a regime where it should be working, so the residuals we see are attributable to
specific pipeline components rather than to a breakdown of the small-signal model.

In [ ]:
A_WAVES    = 0.10       # sinusoid amplitude in waves
NU0        = 10         # spatial frequency, cycles per aperture
DZ_MM      = 40         # defocus diversity distance, mm
N_ITERS    = 150        # FDPR iterations
PHOTONS    = 1e6        # total photon count per PSF
READ_NOISE = 5.0        # e- read noise std

D          = seal_parameters['pupil_size']
wavelength = seal_parameters['wavelength_meter']
f_len      = seal_parameters['focal_length_meters']
rad_to_nm  = wavelength * 1e9 / (2 * np.pi)

## Build Truth Aberration and Defocus Diversity

Construct the pupil-plane truth $\phi(x, y) = A \sin(2 \pi \nu_0 x)$ in radians
(so the peak-to-valley wavefront error is $2A$ waves), plus the diversity defocus
$\phi_{\Delta z}$ from Eq.~\ref{eq:defocus_phase} of the paper.  Both fields
share the pupil grid and are multiplied by the SEAL aperture mask before
propagation; this keeps the reconstruction problem matched to the
forward model and avoids the off-aperture phase the FDPR algorithm has no
information about.

In [ ]:
# Sinusoidal aberration across the pupil (radians)
phi_ab = Field(A_WAVES * 2 * np.pi * np.sin(2 * np.pi * NU0 * pupil_grid.x / D), pupil_grid)

# Scale Zernike defocus template to match the requested DZ_MM
# delta = 8 * P * (f/D)^2  =>  P = delta / (8*(f/D)^2)  (meters)
p2v_tmpl     = float(np.max(defocus_tmpl) - np.min(defocus_tmpl))  # rad P2V of unit-Zernike
target_P_m   = abs(DZ_MM * 1e-3) / (8 * (f_len / D)**2)
target_P_rad = target_P_m * 2 * np.pi / wavelength
defocus_scale = target_P_rad / p2v_tmpl
phi_df = defocus_tmpl * defocus_scale

print(f"Aberration amplitude : {A_WAVES:.2f} waves  ({A_WAVES*2*np.pi*rad_to_nm:.1f} nm RMS-ish)")
print(f"Defocus scale factor : {defocus_scale:.3f}")
print(f"Defocus P2V          : {target_P_rad:.2f} rad  ({target_P_m*1e9:.0f} nm)")

# Quick-look: truth aberration inside the pupil
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(np.array(phi_ab.shaped) * np.array(telescope_pupil.shaped),
                      cmap='RdBu_r')
axes[0].set_title('Truth aberration (rad)'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(np.array(phi_df.shaped) * np.array(telescope_pupil.shaped),
                      cmap='RdBu_r')
axes[1].set_title('Diversity defocus (rad)'); plt.colorbar(im1, ax=axes[1])
plt.tight_layout()

## Generate PSFs (Focused + Diversity) with Noise

Build the three focal-plane intensities $\{I_0, I_{+\Delta z}, I_{-\Delta z}\}$
using the same Fraunhofer forward model the retrieval uses internally; this keeps the
data-vs-model comparison self-consistent and isolates noise as the only source of
discrepancy at this stage.  Inject Poisson photon noise at $N_\gamma = 10^{6}$
photons/frame and Gaussian read noise at $\sigma_r = 11\,e^{-}$ rms, matching the
SEAL detector defaults; the resulting per-pixel variance is the quadrature sum of
Eq.~\ref{eq:pixel_variance}.  We deliberately stay below the per-pixel saturation
regime so the noise budget is read-limited at the OTF-pixel level
(Section~\ref{sec:noise_budget}); pushing into the photon-limited corner is left to
the photon-count sweep further down this notebook.

In [ ]:
def make_wf(phase_field):
    """Create a Wavefront with given phase applied to the telescope pupil."""
    wf = Wavefront(telescope_pupil, wavelength=wavelength)
    wf.electric_field = telescope_pupil * np.exp(1j * phase_field)
    return wf

def make_psf(phase_field):
    """Propagate phase to focal plane, return intensity as 2D numpy array."""
    focal = prop_p2f(make_wf(phase_field))
    return np.array(focal.intensity.shaped)

def add_noise(psf, total_photons, read_noise):
    """Add shot noise (Poisson) and Gaussian read noise."""
    scaled = psf / psf.sum() * total_photons
    shot = np.random.poisson(np.clip(scaled, 0, None).astype(float))
    noisy = shot + np.random.normal(0, read_noise, psf.shape)
    return np.clip(noisy, 0, None)

# Noiseless PSFs
psf_f_clean = make_psf(phi_ab)
psf_p_clean = make_psf(phi_ab + phi_df)
psf_m_clean = make_psf(phi_ab - phi_df)

# Noisy PSFs
np.random.seed(42)
psf_f = add_noise(psf_f_clean, PHOTONS, READ_NOISE)
psf_p = add_noise(psf_p_clean, PHOTONS, READ_NOISE)
psf_m = add_noise(psf_m_clean, PHOTONS, READ_NOISE)

# Quick-look
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(axes,
                           [psf_f, psf_p, psf_m],
                           ['Focused', f'+{DZ_MM} mm defocus', f'-{DZ_MM} mm defocus']):
    ax.imshow(np.log10(img / img.max() + 1e-12), vmin=-5, cmap='inferno')
    ax.set_title(title); ax.axis('off')
plt.tight_layout()

## Run FDPR

Run the Misell-style amplitude-projection FDPR loop (Section~\ref{sec:convergence})
on the three-image stack for $N_{\rm iter} = 150$ steps with a random phase
initialization.  The 3-image configuration $(I_0, I_{+\Delta z}, I_{-\Delta z})$
is the minimum multi-image geometry that carries enough defocus signal for
Dean-Bowers selection while remaining symmetric; we use it as the reference
throughout the paper.  The iteration runs to a fixed budget rather than a
tolerance criterion (the latter is identified as future work in
Section~\ref{sec:discussion}), so $N_{\rm iter}$ has to be set large enough that the
recon is at the small-signal fixed point at the operating point chosen above ---
the iteration sweep below verifies this empirically.

In [ ]:
DZ_UM = DZ_MM * 1e3  # mm -> microns for FDPR

mp = FocusDiversePhaseRetrieval(
    [psf_f, psf_p, psf_m],
    WAVELENGTH_UM,
    [IMAGE_DX_UM, IMAGE_DX_UM],
    [DZ_UM, -DZ_UM],
)

for _ in range(N_ITERS):
    psf_rec = mp.step()

# Cost function convergence
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(mp.cost_functions[0], label=f'+{DZ_MM} mm')
ax.semilogy(mp.cost_functions[1], label=f'-{DZ_MM} mm')
ax.set_xlabel('Iteration'); ax.set_ylabel('MSE')
ax.set_title('FDPR Cost Function Convergence')
ax.legend(); plt.tight_layout()

### Reading the cost-function plot

Both `+Δz` and `-Δz` curves should drop monotonically and plateau.

- **Smooth monotonic decrease, asymptote within ~50 iter:** the iteration is at its fixed
  point; you are in the regime the rest of this notebook assumes.
- **Curve still descending steeply at 150 iter:** the budget is too small; either raise
  `N_ITERS` or move to a tolerance-based stop (identified as future work in the paper).
- **Oscillations or non-monotonic descent:** the amplitude-projection loop is bouncing
  between near-degenerate basins. The seed test at the end of the notebook will tell you
  whether this is reproducible across initializations.
- **`-Δz` curve much higher than `+Δz`:** asymmetric encoding in the diversity pair.
  Re-check the sign convention on `phi_df`; the Dean-Bowers theory assumes
  $|\Delta z_+| = |\Delta z_-|$.

## Extract Recon Phase + Truth / Recon / Residual RMS

Take the angle of the MFT-reversed focused complex amplitude as the recon phase,
piston-subtract both truth and recon by the aperture-median (which coincides with
the aperture mean for the sinusoid and coma modes used here; bases with nonzero
aperture mean would need this convention revisited), and compute the residual RMS
over illuminated pupil pixels as in Eq.~\ref{eq:rms_residual}.  Three numbers come
out of this cell: the truth RMS (a property of the injected aberration), the recon
RMS (a property of what the algorithm returns), and the residual RMS (the
quantity that actually matters for diversity-selection scoring).  The relative
ordering of these three is the first diagnostic --- residual $<$ truth means the
retrieval beats the null, residual $>$ truth means the retrieval has degraded past
the point of improvement and the small-signal model has broken down for this
operating point.

In [ ]:
# Reconstruct pupil phase from FDPR output
raw_recon  = np.angle(mft_rev(psf_rec, conf))
recon_full = resize(raw_recon, (256, 256), preserve_range=True) * telescope_pupil.shaped

# Truth phase on the same 256x256 grid
truth_full = np.array(phi_ab.shaped) * np.array(telescope_pupil.shaped)

# Piston-subtracted masked vectors
truth_ms = truth_full[pupil_mask] - np.median(truth_full[pupil_mask])
rec_ms   = recon_full[pupil_mask] - np.median(recon_full[pupil_mask])

def rms_nm(x):
    return float(np.sqrt(np.nanmean(x**2))) * rad_to_nm

truth_rms = rms_nm(truth_ms)
recon_rms = rms_nm(rec_ms)
resid_rms = rms_nm(truth_ms - rec_ms)

print(f"truth_rms  = {truth_rms:.2f} nm")
print(f"recon_rms  = {recon_rms:.2f} nm")
print(f"resid_rms  = {resid_rms:.2f} nm")
print(f"resid > recon? {resid_rms > recon_rms}")

# Side-by-side maps
truth_map = np.full((256, 256), np.nan); truth_map[pupil_mask] = truth_ms * rad_to_nm
recon_map = np.full((256, 256), np.nan); recon_map[pupil_mask] = rec_ms * rad_to_nm
resid_map = np.full((256, 256), np.nan); resid_map[pupil_mask] = (truth_ms - rec_ms) * rad_to_nm

vmax = float(np.nanmax(np.abs(truth_map)))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, m, t in zip(axes,
                     [truth_map, recon_map, resid_map],
                     [f'Truth  {truth_rms:.1f} nm',
                      f'Recon  {recon_rms:.1f} nm',
                      f'Residual  {resid_rms:.1f} nm']):
    im = ax.imshow(m, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(t); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()

### Reading `truth_rms / recon_rms / resid_rms`

Three numbers, in nm of wavefront error.  The relative ordering tells the failure mode:

- **`resid_rms < recon_rms < truth_rms`:** retrieval beats the null, in a healthy
  amplitude-bias regime ($\alpha > 0.5$).  Production sweeps should produce this.
- **`recon_rms < truth_rms` and `resid_rms > recon_rms`** (the case here, 22 vs 27 nm
  with truth 46 nm): geometric consequence of $\alpha < 0.5$.  Not a bug; the next cell
  separates the bias piece from the noise piece to prove it.
- **`recon_rms ≈ 0` and `resid_rms ≈ truth_rms`:** retrieval has collapsed to a near-zero
  phase ($\alpha \to 0$).  Either the noise budget is too thin, or the operating point
  is past the cutoff documented by the frequency sweep further down.
- **`resid_rms > truth_rms`:** retrieval is *worse* than the null.  This happens at
  $A \sim \pi$ rad (Section~\ref{sec:mc_sweeps} of the paper) when the small-signal
  expansion no longer describes the focal-plane intensity.

## Alpha Decomposition: Amplitude Bias vs Orthogonal Noise

Decompose the recon as $\phi_{\rm rec} = \alpha\,\phi_{\rm truth} + \delta$,
where $\alpha = \langle \phi_{\rm rec}, \phi_{\rm truth}\rangle / \|\phi_{\rm truth}\|^2$
is the scalar projection onto the truth direction and $\delta$ is the orthogonal
remainder by construction.  The point of separating $\alpha$ from $\delta$ is that
the two failure modes that show up in §3.5 of the paper (small-signal saturation
and noise-driven scatter) live in different places in this decomposition:
amplitude-bias / linearization saturation lives entirely in $\alpha$, while
noise, modal cross-talk, MFT sampling, and `skimage` resize artifacts live entirely
in $\delta$.  A reconstruction with $\alpha = 0.86$ and $\delta = 0$ would still
report a large residual RMS even though the algorithm is recovering the truth
direction perfectly --- residual $>$ recon whenever $\alpha < 0.5$ is the
geometric consequence, not a bug.

In [ ]:
# 1-D projection: recon onto truth
alpha = np.sum(rec_ms * truth_ms) / np.sum(truth_ms * truth_ms)
parallel   = alpha * truth_ms            # truth-aligned component of recon
orthogonal = rec_ms - parallel            # everything else in recon

# Decompose residual
resid_parallel   = (1.0 - alpha) * truth_ms   # predicted from amplitude bias alone
resid_orthogonal = -orthogonal                 # noise + modal cross-talk + edge

print(f"alpha (amplitude bias)         = {alpha:.3f}")
print(f"  crossover at 0.5 => resid {'>' if alpha < 0.5 else '<='} recon")
print()
print(f"truth_rms                      = {truth_rms:.2f} nm")
print(f"recon_rms                      = {recon_rms:.2f} nm")
print(f"  parallel piece (alpha*truth) = {rms_nm(parallel):.2f} nm")
print(f"  orthogonal piece (delta)     = {rms_nm(orthogonal):.2f} nm")
print()
print(f"resid_rms (measured)           = {resid_rms:.2f} nm")
print(f"  predicted (1-alpha)*truth    = {rms_nm(resid_parallel):.2f} nm")
print(f"  unexplained (delta)          = {rms_nm(resid_orthogonal):.2f} nm")
print()
# Sanity: quadrature should match measured residual
predicted_resid = np.sqrt(rms_nm(resid_parallel)**2 + rms_nm(resid_orthogonal)**2)
print(f"quadrature check               = {predicted_resid:.2f} nm  (should ~ {resid_rms:.2f})")

### Reading the alpha decomposition

The point of this cell is the quadrature check at the bottom: the measured residual
RMS should equal $\sqrt{((1-\alpha)\phi_{\rm truth})^2 + \delta^2}$ to within
floating-point precision.  Once that check passes, $\alpha$ alone tells you which
failure mode you are in.

| $\alpha$ value | Interpretation |
|----|----|
| $\approx 1.0$ | Truth fully recovered; residual is pure noise + pipeline floor. |
| $\approx 0.85$ | **Pipeline ceiling** -- $\alpha_{\rm clean}$ for this operating point. Even infinite photons cannot exceed this; the gap from 0.85 to 1.0 is MFT/resize/centering. |
| $0.5 - 0.85$ | Noise-induced amplitude bias.  More photons (or a Poisson-MLE algorithm) will raise $\alpha$; pipeline-floor still applies. |
| $\approx 0.5$ | The geometric crossover: $\text{resid\_rms} = \text{recon\_rms}$.  No deeper meaning, but useful to anchor the plots. |
| $0 - 0.5$ | Severe noise-induced bias.  Recon is a faithful-but-attenuated copy of the truth. |
| $\approx 0$ | Past the cutoff.  Recon is uncorrelated with truth. |
| Negative | Recon is *anti-correlated* with truth.  Almost always means a sign or basin-flip in the diversity, not signal. |

The split between $\alpha\,\phi_{\rm truth}$ and $\delta$ is the diagnostic that lets you
ask "is more exposure going to help?" -- the answer is *only* if $\alpha$ is what's killing
you and you are still on the rising part of the photon-count curve.

## Project Delta onto Tip / Tilt / Defocus

The orthogonal $\delta$ contains everything that isn't a scaled copy of the truth;
the question is whether that "everything" is incoherent noise or whether it carries
coherent low-order modes that the retrieval is leaking into.  We project $\delta$
onto $\{x, y, x^2 + y^2\}$ (tip, tilt, defocus) on the masked pupil and report the
fraction of $\delta$'s power each mode explains.  A non-trivial tip/tilt
component would indicate a pixel-centering inconsistency between the forward
model and the recon; a non-trivial defocus component would indicate that the
diversity defocus is being absorbed into the unknown phase, which is the
classic sign of a sign-degeneracy not being broken cleanly by the chosen
$\Delta z$.

In [ ]:
# Basis functions on the masked pupil
xg = np.array(pupil_grid.x)[pupil_mask.ravel()]
yg = np.array(pupil_grid.y)[pupil_mask.ravel()]
xg = xg - np.mean(xg)
yg = yg - np.mean(yg)

def proj(target, basis):
    """Project target onto a single basis vector (mean-subtracted)."""
    b = basis - np.mean(basis)
    return np.sum(target * b) / np.sum(b * b)

# Tip / tilt
c_tip  = proj(resid_orthogonal, xg)
c_tilt = proj(resid_orthogonal, yg)
tt_part = c_tip * xg + c_tilt * yg

# Defocus leak (recon picked up some Z4 from the diversity)
z4 = np.array(zernike_modes[3].shaped)[pupil_mask]
z4 = z4 - np.median(z4)
c_def = proj(resid_orthogonal, z4)
def_part = c_def * z4

# High-frequency residual (what's left)
hf = resid_orthogonal - tt_part - def_part

print(f"orthogonal delta total rms = {rms_nm(resid_orthogonal):.2f} nm")
print(f"  tip/tilt rms             = {rms_nm(tt_part):.2f} nm")
print(f"  defocus  rms             = {rms_nm(def_part):.2f} nm")
print(f"  high-freq residual rms   = {rms_nm(hf):.2f} nm")

### Reading the tip / tilt / defocus split

The orthogonal $\delta$ is split into low-order coherent modes (tip, tilt, defocus) and
a "high-frequency residual" lump.  What you want to see is **the low-order pieces being
small** relative to the high-frequency lump.

- **tt rms / total $\delta$ rms $\lesssim 10\%$ (the case here, 0.43/9.09):** clean.
  No pixel-centering inconsistency between forward model and recon.  Move on.
- **tt rms $\gtrsim$ 5 nm and growing with frequency:** the forward model and the recon
  disagree on where the pupil center is.  Inspect `pupil_grid` and the MFT-reverse
  step.  This often shows up alongside a non-zero $\phi_0$ (see end of notebook).
- **Defocus rms $\gtrsim$ 5 nm:** the diversity defocus is leaking into the unknown
  phase, i.e.\ the sign degeneracy is not being broken cleanly.  Either $|\Delta z|$ is
  too small for the current $\nu_0$ (consult Section~\ref{sec:otf_heatmap}), or the
  $+\Delta z / -\Delta z$ pair is asymmetric.
- **High-frequency residual dominant:** expected.  This is incoherent noise plus
  pixel-scale edge effects, and is what the $\delta$ piece is supposed to absorb.

## Iteration Sweep: Is Alpha Still Climbing?

Run FDPR for $N_{\rm iter} = 300$ rather than 150 and log $\alpha$, the residual
RMS, and the recon RMS every 10 steps.  Three patterns can show up:

- **$\alpha$ saturates below 1.0**: the small-signal model itself is the floor;
  more iterations cannot help, and the gap from $\alpha_\infty$ to 1 must be
  attributed to pipeline geometry (MFT sampling, resize interpolation, edge
  effects) rather than to the retrieval.
- **$\alpha$ still climbing at 150**: 150 iterations is undersampling the
  fixed point; the production sweeps need to be rerun at a larger budget
  (try 500 or 1000) or with a tolerance criterion.
- **$\alpha$ oscillates**: the cost landscape has multiple basins and the
  amplitude-projection loop is hopping between them; an MLE formulation
  (Section~\ref{sec:discussion} of the paper) is the standard fix.

In [ ]:
SWEEP_ITERS = 300  # extend beyond the original N_ITERS to see if alpha keeps climbing
SAMPLE_EVERY = 10

mp_sweep = FocusDiversePhaseRetrieval(
    [psf_f, psf_p, psf_m],
    WAVELENGTH_UM,
    [IMAGE_DX_UM, IMAGE_DX_UM],
    [DZ_UM, -DZ_UM],
)

hist = []
for k in range(SWEEP_ITERS):
    psf_rec_k = mp_sweep.step()
    if (k + 1) % SAMPLE_EVERY == 0:
        raw_k = np.angle(mft_rev(psf_rec_k, conf))
        rec_full_k = resize(raw_k, (256, 256), preserve_range=True) * telescope_pupil.shaped
        rec_ms_k = rec_full_k[pupil_mask] - np.median(rec_full_k[pupil_mask])
        a_k = float(np.sum(rec_ms_k * truth_ms) / np.sum(truth_ms**2))
        hist.append((k + 1, a_k, rms_nm(rec_ms_k), rms_nm(truth_ms - rec_ms_k)))

hist = np.array(hist)
print("iter   alpha   recon_rms  resid_rms")
for row in hist:
    print(f"{int(row[0]):4d}   {row[1]:.3f}   {row[2]:7.2f}    {row[3]:7.2f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(hist[:, 0], hist[:, 1], 'o-', markersize=3)
ax1.axhline(0.5, ls='--', color='gray', label=r'$\alpha=0.5$ crossover')
ax1.set_xlabel('Iteration'); ax1.set_ylabel(r'$\alpha$')
ax1.set_title(r'Amplitude bias $\alpha$ vs iteration'); ax1.legend()

ax2.plot(hist[:, 0], hist[:, 2], 'o-', markersize=3, label='recon RMS')
ax2.plot(hist[:, 0], hist[:, 3], 's-', markersize=3, label='resid RMS')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('RMS (nm)')
ax2.set_title('Recon & Residual RMS vs iteration'); ax2.legend()

plt.tight_layout()

### Reading the iteration sweep

Three diagnostic patterns to look for in the $\alpha$-vs-iteration plot:

- **$\alpha$ plateaus within 100-150 iter:** the 150-iter budget the paper uses is fine
  for this operating point.  The fixed point is reached.
- **$\alpha$ still climbing at iter 300:** the budget is undersampled.  Either raise
  `N_ITERS` for production sweeps, or move to a tolerance-based stop.  The 150-iter
  budget will produce numerically-different residuals than a 300- or 500-iter budget.
- **$\alpha$ oscillates between two values:** the cost landscape has two near-degenerate
  basins and the amplitude-projection loop is hopping between them.  The seed test at the
  end of the notebook tells you whether the oscillation is reproducible (basin choice
  determined by initialization) or genuine non-convergence.  In either case the
  Poisson-MLE formulation (Section~\ref{sec:discussion} of the paper) is the standard fix.

The recon RMS and residual RMS plots are the same information in physical units: as
$\alpha$ rises toward its asymptote, the recon RMS approaches the truth RMS and the
residual approaches the orthogonal-only floor.

## Noiseless Comparison: Pipeline Floor vs Noise Contribution

Repeat the same retrieval on noiseless PSFs.  The orthogonal $\delta$ from the
clean run is the *pipeline floor* --- MFT sampling, resize interpolation, and any
even/odd grid asymmetries propagate through the noiseless forward model and end up
in $\delta$ regardless of how many photons we collect.  Adding noise inflates
$\delta$ in quadrature, so $\delta_{\rm noisy}^2 - \delta_{\rm clean}^2$
is the noise contribution.  This is the cleanest way to ask whether the residuals
in §3.5 of the paper are dominated by the pipeline or by the detector budget; if
$\delta_{\rm clean}$ is already comparable to the production residual floor,
expanding the photon budget will not help.

In [ ]:
mp_clean = FocusDiversePhaseRetrieval(
    [psf_f_clean, psf_p_clean, psf_m_clean],
    WAVELENGTH_UM,
    [IMAGE_DX_UM, IMAGE_DX_UM],
    [DZ_UM, -DZ_UM],
)
for _ in range(N_ITERS):
    psf_rec_clean = mp_clean.step()

# Extract clean recon
raw_clean  = np.angle(mft_rev(psf_rec_clean, conf))
recon_clean = resize(raw_clean, (256, 256), preserve_range=True) * telescope_pupil.shaped
rec_ms_clean = recon_clean[pupil_mask] - np.median(recon_clean[pupil_mask])

# Clean alpha decomposition
alpha_clean = float(np.sum(rec_ms_clean * truth_ms) / np.sum(truth_ms**2))
parallel_clean   = alpha_clean * truth_ms
orthogonal_clean = rec_ms_clean - parallel_clean

# Compare noisy vs clean
delta_noisy_rms = rms_nm(orthogonal)
delta_clean_rms = rms_nm(orthogonal_clean)
delta_noise_only = np.sqrt(max(delta_noisy_rms**2 - delta_clean_rms**2, 0))

print(f"--- Noiseless run ---")
print(f"alpha (clean)       = {alpha_clean:.3f}   (noisy was {alpha:.3f})")
print(f"recon_rms (clean)   = {rms_nm(rec_ms_clean):.2f} nm")
print(f"resid_rms (clean)   = {rms_nm(truth_ms - rec_ms_clean):.2f} nm")
print()
print(f"--- Delta breakdown ---")
print(f"delta_rms (noisy)   = {delta_noisy_rms:.2f} nm")
print(f"delta_rms (clean)   = {delta_clean_rms:.2f} nm  <-- pipeline floor")
print(f"delta_rms (noise)   = {delta_noise_only:.2f} nm  <-- noise contribution (quadrature)")

### Reading the noiseless vs noisy delta breakdown

This is the most consequential cell in the notebook for interpreting the
production residuals.  Three numbers come out:

- **`alpha (clean)` vs `alpha (noisy)`:** the gap is the noise-induced amplitude
  bias.  A 0.86 -> 0.43 collapse (as here) means the noise is robbing about half of the
  amplitude even though it is invisible in the orthogonal direction.  This is a known
  failure mode of amplitude-projection algorithms and is one of the reasons the paper
  identifies a Poisson-MLE rewrite as future work.
- **`delta_rms (clean)`:** the **pipeline floor**.  MFT sampling, `skimage` resize, and
  any odd/even centering asymmetries propagate through the noiseless forward model and
  are stuck in $\delta$ regardless of photon count.  Production residuals can never go
  below this number at this operating point.
- **`delta_rms (noise)` via quadrature:** if this is $\approx 0$ (as here, 0.0 nm), the
  detector noise is invisible in the orthogonal direction at the current $N_\gamma$ and
  $\sigma_r$.  Adding more photons will fix $\alpha$ but will not reduce $\delta$ further.
  If the quadrature is large compared to `delta_clean`, you are in the photon-noise-limited
  regime and lowering $\sigma_r$ or raising $N_\gamma$ both help.

**Practical rule:** if `delta_clean` $\gtrsim$ `truth_rms / 10`, the pipeline floor is the
dominant residual and the bottleneck is the recon-side propagation, not the data.

## 4-Panel Decomposition Figure

Plot, side by side, the truth phase, the recon phase, the parallel component
$\alpha \phi_{\rm truth}$, and the orthogonal remainder $\delta$, all on the
same color scale.

- If panel 4 (the orthogonal $\delta$) looks like high-frequency noise plus a
  thin ring around the aperture edge, the model is confirmed: residual energy
  is incoherent + edge-effect.
- If panel 4 shows coherent structure --- a tilted plane, a defocus bowl, or a
  sinusoid at a different $\nu_0$ than truth --- there is a real
  cross-coupling worth chasing in a follow-up, because that energy is in
  principle removable by adjusting the basis or the diversity.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

mvmax = float(np.nanmax(np.abs(truth_map)))

# Build 2D maps from masked vectors
def vec_to_map(vec):
    m = np.full((256, 256), np.nan)
    m[pupil_mask] = vec * rad_to_nm
    return m

maps = [vec_to_map(truth_ms),
        vec_to_map(rec_ms),
        vec_to_map(parallel),
        vec_to_map(orthogonal)]

titles = [
    f"Truth\n{truth_rms:.1f} nm",
    f"Recon\n{recon_rms:.1f} nm",
    rf"$\alpha \cdot$ Truth ($\alpha$={alpha:.2f})" + f"\n{rms_nm(parallel):.1f} nm",
    f"Orthogonal $\\delta$\n{rms_nm(orthogonal):.1f} nm",
]

for ax, m, t in zip(axes, maps, titles):
    im = ax.imshow(m, cmap='RdBu_r', vmin=-mvmax, vmax=mvmax)
    ax.set_title(t); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

fig.tight_layout()
fig.savefig('fdpr_residual_decomp.png', dpi=150, bbox_inches='tight')
print("Saved fdpr_residual_decomp.png")

### Reading the four-panel decomposition figure

This is a sanity check on the alpha decomposition rather than a quantitative tool.
The four maps are on a common color scale so they can be compared directly.

- **Panel 1 (truth):** the injected sinusoid.  Visible structure depends on $\nu_0$ and
  $A$, but should look like a clean grating.
- **Panel 2 (recon):** the algorithm output.  At healthy operating points it should be
  a visibly attenuated version of panel 1 with no extra features.
- **Panel 3 ($\alpha\cdot$ truth):** what the recon would look like if it were a *pure*
  amplitude-biased copy of the truth.  This is the "model prediction" for the recon.
- **Panel 4 ($\delta$, orthogonal):** the part of the recon that is *not* a scaled copy
  of the truth.  This is the diagnostic panel.

  - **Looks like incoherent noise with a thin ring at the aperture edge:** model
    confirmed.  Residual energy is pipeline floor + noise, no coherent leakage.
  - **Has a visible tilted plane or a defocus bowl:** modal cross-talk into low-order
    Zernikes.  Inspect the tip/tilt/defocus split numbers above; the projection should
    have caught it.
  - **Has a visible sinusoid at a different $\nu_0$ than truth:** aliasing.  Either the
    truth is near or past the practical cutoff, or the MFT grid is undersampling the
    defocused PSF.

## Frequency Cutoff Diagnostic: Where Does Recovery Break Down?

Sweep $\nu_0$ from 2 to 40 cyc/aperture at fixed $A$ and $\Delta z$ and track
$\alpha$, the residual RMS, and the recon RMS at each frequency.  This is the
diagnostic that motivates the $\nu_0 \geq 5$ cyc/aperture lower bound on the
regime in which Section~\ref{sec:discussion} of the paper reports the OTF / FDPR
correlation; below 5 cyc/aperture the sinusoid resembles a low-order Zernike that
the focused PSF resolves directly, and above the cutoff the retrieval just stops
working.

The Nyquist rule of thumb sets the absolute upper limit at 128 cyc/aperture for a
256-pixel pupil, but the *practical* FDPR cutoff is much lower --- expected near
15 cyc/aperture given the MFT focal-plane sampling and the `skimage` resize step
that downsamples the focal-plane onto the pupil grid.  The point of the sweep is
to measure that cutoff rather than estimate it.

In [ ]:
freq_list = [2, 4, 6, 8, 10, 12, 14, 15, 16, 18, 20, 25, 30, 40]
FREQ_ITERS = 150

freq_results = []

for nu in freq_list:
    # Build sinusoidal truth at this frequency
    phi_ab_nu = Field(A_WAVES * 2 * np.pi * np.sin(2 * np.pi * nu * pupil_grid.x / D), pupil_grid)
    truth_nu  = np.array(phi_ab_nu.shaped) * np.array(telescope_pupil.shaped)
    t_ms_nu   = truth_nu[pupil_mask] - np.median(truth_nu[pupil_mask])

    # Forward-model PSFs (noiseless for clean comparison)
    pf = make_psf(phi_ab_nu)
    pp = make_psf(phi_ab_nu + phi_df)
    pm = make_psf(phi_ab_nu - phi_df)

    # Add noise
    np.random.seed(42)
    pf_n = add_noise(pf, PHOTONS, READ_NOISE)
    pp_n = add_noise(pp, PHOTONS, READ_NOISE)
    pm_n = add_noise(pm, PHOTONS, READ_NOISE)

    # Run FDPR
    mp_nu = FocusDiversePhaseRetrieval(
        [pf_n, pp_n, pm_n], WAVELENGTH_UM,
        [IMAGE_DX_UM, IMAGE_DX_UM], [DZ_UM, -DZ_UM])
    for _ in range(FREQ_ITERS):
        rec_nu = mp_nu.step()

    # Extract recon
    raw_nu   = np.angle(mft_rev(rec_nu, conf))
    recon_nu = resize(raw_nu, (256, 256), preserve_range=True) * telescope_pupil.shaped
    r_ms_nu  = recon_nu[pupil_mask] - np.median(recon_nu[pupil_mask])

    # Metrics
    a_nu     = float(np.sum(r_ms_nu * t_ms_nu) / np.sum(t_ms_nu**2))
    t_rms_nu = rms_nm(t_ms_nu)
    r_rms_nu = rms_nm(r_ms_nu)
    e_rms_nu = rms_nm(t_ms_nu - r_ms_nu)

    freq_results.append((nu, a_nu, t_rms_nu, r_rms_nu, e_rms_nu))
    print(f"nu0={nu:3d}  alpha={a_nu:.3f}  truth={t_rms_nu:.1f}  recon={r_rms_nu:.1f}  resid={e_rms_nu:.1f} nm")

freq_results = np.array(freq_results)

### Reading the alpha-vs-frequency table

This is the diagnostic that pins down the practical FDPR cutoff at this aberration
amplitude, photon budget, and read noise.  Three regimes are visible:

| $\alpha$ range | Regime | What it means |
|----|----|----|
| $\alpha > 0.85$ | Pipeline-ceiling regime | Algorithm at the geometric limit; residual is mostly the pipeline floor.  Operating points to use for cross-validation. |
| $0.4 < \alpha < 0.85$ | Noise-bias regime | Recon recovers the truth direction with severe amplitude bias.  More photons will move you up this band. |
| $|\alpha| < 0.2$ | Past-cutoff regime | Recon decorrelates from truth.  Recon RMS $\to$ noise floor (~5 nm), residual RMS $\to$ truth RMS. |
| $\alpha < 0$ | Anti-correlated | Recon is the *negative* of truth.  Algorithm picked the wrong basin; almost always a sign issue with the diversity. |

The transition is **sharp** -- $\alpha$ drops from 0.88 at $\nu_0=8$ to 0.43 at
$\nu_0=10$, and from 0.57 at $\nu_0=15$ to $-0.12$ at $\nu_0=16$.  These two thresholds
($\nu_0 \approx 8$ and $\nu_0 \approx 15$) bound the band in which a noisy single-cell
retrieval has any chance of producing the right answer at this $\Delta z$.  Below 8
c/aperture and above 15 c/aperture, the per-cell retrieval is decoupled from the OTF
sideband prediction for the reasons discussed in Section~\ref{sec:discussion} of the paper.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

nu = freq_results[:, 0]

# --- Panel 1: alpha vs frequency ---
ax1.plot(nu, freq_results[:, 1], 'o-', color='tab:blue', markersize=5)
ax1.axhline(0.5, ls='--', color='gray', alpha=0.6, label=r'$\alpha=0.5$ (resid=recon crossover)')
ax1.axvline(15, ls=':', color='red', alpha=0.7, label='$\\nu_0 = 15$ c/ap')
ax1.set_xlabel('Spatial frequency $\\nu_0$ (cycles/aperture)')
ax1.set_ylabel(r'$\alpha$ (amplitude recovery)')
ax1.set_title(r'Recovery amplitude $\alpha$ vs frequency')
ax1.set_ylim(-0.1, 1.1)
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Panel 2: RMS values vs frequency ---
ax2.plot(nu, freq_results[:, 2], 's--', color='black', markersize=4, label='truth RMS')
ax2.plot(nu, freq_results[:, 3], 'o-', color='tab:blue', markersize=4, label='recon RMS')
ax2.plot(nu, freq_results[:, 4], '^-', color='tab:red', markersize=4, label='residual RMS')
ax2.axvline(15, ls=':', color='red', alpha=0.7, label='$\\nu_0 = 15$ c/ap')
ax2.set_xlabel('Spatial frequency $\\nu_0$ (cycles/aperture)')
ax2.set_ylabel('RMS (nm)')
ax2.set_title('Truth / Recon / Residual RMS vs frequency')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig('fdpr_freq_cutoff.png', dpi=150, bbox_inches='tight')
print("Saved fdpr_freq_cutoff.png")

## Before / After Cutoff: Visual Comparison

Plot truth vs recon maps at five frequencies straddling the ~15 cyc/aperture
cutoff measured above ($\nu_0 \in \{6, 10, 15, 20, 30\}$).  The eye picks up the
failure mode that the scalar $\alpha$ averages over: below the cutoff the recon
is a clean copy of the truth, at the cutoff it picks up an amplitude bias but the
spatial pattern is right, and above the cutoff the recon develops a
visibly-different frequency or collapses to noise.  This is the visual analog of
the per-frequency $\alpha$ that the next plot shows quantitatively.

In [ ]:
showcase_freqs = [6, 10, 15, 20, 30]
n_show = len(showcase_freqs)

fig, axes = plt.subplots(3, n_show, figsize=(4 * n_show, 11))

for j, nu_s in enumerate(showcase_freqs):
    # Rebuild truth + run FDPR at this frequency
    phi_s   = Field(A_WAVES * 2 * np.pi * np.sin(2 * np.pi * nu_s * pupil_grid.x / D), pupil_grid)
    truth_s = np.array(phi_s.shaped) * np.array(telescope_pupil.shaped)
    t_ms_s  = truth_s[pupil_mask] - np.median(truth_s[pupil_mask])

    np.random.seed(42)
    pf_s = add_noise(make_psf(phi_s), PHOTONS, READ_NOISE)
    pp_s = add_noise(make_psf(phi_s + phi_df), PHOTONS, READ_NOISE)
    pm_s = add_noise(make_psf(phi_s - phi_df), PHOTONS, READ_NOISE)

    mp_s = FocusDiversePhaseRetrieval(
        [pf_s, pp_s, pm_s], WAVELENGTH_UM,
        [IMAGE_DX_UM, IMAGE_DX_UM], [DZ_UM, -DZ_UM])
    for _ in range(FREQ_ITERS):
        rec_s = mp_s.step()

    raw_s   = np.angle(mft_rev(rec_s, conf))
    recon_s = resize(raw_s, (256, 256), preserve_range=True) * telescope_pupil.shaped
    r_ms_s  = recon_s[pupil_mask] - np.median(recon_s[pupil_mask])

    a_s = float(np.sum(r_ms_s * t_ms_s) / np.sum(t_ms_s**2))

    # Maps
    t_map = np.full((256, 256), np.nan); t_map[pupil_mask] = t_ms_s * rad_to_nm
    r_map = np.full((256, 256), np.nan); r_map[pupil_mask] = r_ms_s * rad_to_nm
    d_map = np.full((256, 256), np.nan); d_map[pupil_mask] = (t_ms_s - r_ms_s) * rad_to_nm

    vm = float(np.nanmax(np.abs(t_map)))

    for i, (m, label) in enumerate(zip(
            [t_map, r_map, d_map],
            ['Truth', f'Recon ($\\alpha$={a_s:.2f})', 'Residual'])):
        im = axes[i, j].imshow(m, cmap='RdBu_r', vmin=-vm, vmax=vm)
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(f'$\\nu_0$ = {nu_s} c/ap\n{label}', fontsize=10)
        else:
            axes[i, j].set_title(label, fontsize=10)
        if j == n_show - 1:
            plt.colorbar(im, ax=axes[i, j], fraction=0.046)

# Row labels
for i, lbl in enumerate(['Truth', 'Recon', 'Residual']):
    axes[i, 0].set_ylabel(lbl, fontsize=12, rotation=90, labelpad=10)

fig.suptitle('Sine Wave Recovery: Before & After Cutoff (~15 c/ap)', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig('fdpr_freq_cutoff_maps.png', dpi=150, bbox_inches='tight')
print("Saved fdpr_freq_cutoff_maps.png")

### Reading the before/after-cutoff figure

Visual analog of the table above.  Look at each column:

- **Below the cutoff (e.g.\ $\nu_0 = 6$):** recon panel should be a clean shrunk copy
  of the truth panel.  No phase shift, no broken-up structure.
- **In the bias regime (e.g.\ $\nu_0 = 10$):** recon still has the right grating
  orientation and period, but the amplitude is visibly lower.  $\alpha$ tells you the
  exact ratio.
- **At the cutoff ($\nu_0 = 15$):** recon is still recognizable but starting to develop
  phase-shifted patches.  $\alpha$ is around 0.5 and the residual map has noticeable
  coherent structure.
- **Past the cutoff ($\nu_0 = 20$, 30):** recon panel looks like noise.  No phase
  relationship with truth; $\alpha \to 0$.

If a panel **above** the cutoff shows visible structure that *isn't* truth (e.g.\ a
sinusoid at half the expected period), that is aliasing -- the recon is locking onto a
beat frequency between the truth and the focal-plane sampling, not retrieving the input.

## Map α(N_γ): Photon-Count Sweep

Hold $A = 0.10$ waves, $\nu_0 = 10$ cyc/aperture, and $\Delta z = 40$ mm fixed
and sweep $N_\gamma$ over three decades.  The shot-noise variance scales as
$1/N_\gamma$, so $\alpha$ should rise monotonically with $N_\gamma$ and
asymptote to a noise-floor value $\alpha_\infty$.

The point of the sweep is to distinguish two qualitatively different regimes:

- If $\alpha_\infty \to 1$ as $N_\gamma \to \infty$, then the residual at any
  finite $N_\gamma$ is *all noise* and can be reduced by throwing photons at it.
- If $\alpha_\infty < 1$ at infinite photons --- we measure
  $\alpha_\infty \approx 0.86$ for this operating point --- the residual
  splits into a noise-floor piece and a *pipeline floor* piece, and the gap from
  0.86 to 1.0 is geometric (MFT + resize + edge sampling) rather than budgetary.

In the second case, no amount of integration removes that 14\% of the truth that
the pipeline is systematically losing.  Identifying which factor in the
geometric floor is dominant is the role of the three single-knob tests below.

In [ ]:
photon_list = [1e4, 3e4, 1e5, 3e5, 1e6, 3e6, 1e7]

photon_results = []

for n_phot in photon_list:
    np.random.seed(42)
    pf_ph = add_noise(psf_f_clean, n_phot, READ_NOISE)
    pp_ph = add_noise(psf_p_clean, n_phot, READ_NOISE)
    pm_ph = add_noise(psf_m_clean, n_phot, READ_NOISE)

    mp_ph = FocusDiversePhaseRetrieval(
        [pf_ph, pp_ph, pm_ph], WAVELENGTH_UM,
        [IMAGE_DX_UM, IMAGE_DX_UM], [DZ_UM, -DZ_UM])
    for _ in range(N_ITERS):
        rec_ph = mp_ph.step()

    raw_ph   = np.angle(mft_rev(rec_ph, conf))
    recon_ph = resize(raw_ph, (256, 256), preserve_range=True) * telescope_pupil.shaped
    r_ms_ph  = recon_ph[pupil_mask] - np.median(recon_ph[pupil_mask])

    a_ph = float(np.sum(r_ms_ph * truth_ms) / np.sum(truth_ms**2))
    d_ph = rms_nm(r_ms_ph - a_ph * truth_ms)

    # SNR via quadrature: peak pixel signal / sqrt(peak + read_noise^2)
    peak_signal = psf_f_clean.max() / psf_f_clean.sum() * n_phot
    snr_ph = peak_signal / np.sqrt(peak_signal + READ_NOISE**2)

    photon_results.append((n_phot, snr_ph, a_ph, d_ph,
                           rms_nm(r_ms_ph), rms_nm(truth_ms - r_ms_ph)))
    print(f"N_gamma={n_phot:.0e}  SNR={snr_ph:.0f}  alpha={a_ph:.3f}  "
          f"delta={d_ph:.1f} nm  resid={rms_nm(truth_ms - r_ms_ph):.1f} nm")

photon_results = np.array(photon_results)

### Reading the photon-count sweep

Two qualitatively different behaviors can show up:

- **$\alpha$ rises and saturates well below 1.0** (the case here, $\alpha_\infty \approx 0.85$): the
  pipeline ceiling dominates over noise.  Increasing $N_\gamma$ beyond the saturation
  point yields no improvement; the only way to push past the ceiling is to change
  the recon-side propagation.
- **$\alpha$ rises toward 1.0 without obvious saturation:** the pipeline ceiling is
  at or above 1.0 for this operating point (i.e.\ the geometric residual is negligible),
  and the only limit is noise.  More photons will keep helping.

The $\delta$ curve is the complementary diagnostic.  Two behaviors:

- **$\delta_{\rm rms}$ floors at a finite value (~10 nm here):** that floor is the
  pipeline floor.  More photons do not reduce it.
- **$\delta_{\rm rms}$ keeps falling as $N_\gamma$ grows:** noise is still dominant
  in the orthogonal direction.  Photon-noise-limited regime.

**Reading the SNR column:** SNR is the peak-pixel signal-to-noise of the focused PSF
under the quadrature variance budget.  Operating points with SNR $\lesssim$ 10 are
shot-noise-dominated and almost always produce $\alpha < 0.2$ regardless of operating
point.  SNR $\gtrsim$ 50 is where the bias regime becomes navigable.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

snr_vals = photon_results[:, 1]
alpha_vals = photon_results[:, 2]
delta_vals = photon_results[:, 3]

# --- Left: alpha vs SNR ---
ax1.semilogx(photon_results[:, 0], alpha_vals, 'o-', color='tab:blue', markersize=6)
ax1.axhline(alpha_clean, ls='--', color='tab:orange',
            label=rf'Clean ceiling $\alpha_{{clean}}$={alpha_clean:.3f}')
ax1.axhline(1.0, ls=':', color='gray', alpha=0.5, label=r'$\alpha=1$ (perfect)')
ax1.axhline(0.5, ls='--', color='gray', alpha=0.4, label=r'$\alpha=0.5$ crossover')
ax1.set_xlabel(r'$N_\gamma$ (total photons)')
ax1.set_ylabel(r'$\alpha$')
ax1.set_title(r'Amplitude recovery $\alpha$ vs photon count')
ax1.set_ylim(-0.05, 1.1)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Right: delta_rms vs SNR ---
ax2.loglog(photon_results[:, 0], delta_vals, 's-', color='tab:red', markersize=6,
           label=r'$\delta$ rms (orthogonal)')
ax2.axhline(rms_nm(orthogonal_clean), ls='--', color='tab:orange',
            label=f'Clean floor {rms_nm(orthogonal_clean):.1f} nm')
ax2.set_xlabel(r'$N_\gamma$ (total photons)')
ax2.set_ylabel(r'$\delta$ RMS (nm)')
ax2.set_title('Orthogonal residual vs photon count')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
fig.savefig('fdpr_alpha_vs_photons.png', dpi=150, bbox_inches='tight')
print("Saved fdpr_alpha_vs_photons.png")

### Reading the alpha-vs-photons figure

Two panels, both log-x:

- **Left ($\alpha$ vs $N_\gamma$):**  Look for the dashed *clean-ceiling* line.  If the
  curve asymptotes *to* it, the pipeline is saturated and adding photons buys nothing.
  If the curve is still climbing toward it, you have room.  The dashed line at $\alpha = 0.5$
  is just the geometric crossover (where `resid_rms = recon_rms`).
- **Right ($\delta_{\rm rms}$ vs $N_\gamma$, log-log):** the *clean floor* line is the
  pipeline floor.  A curve that floors at it is pipeline-limited; a curve that stays well
  above it (with negative slope) is noise-limited.  A $1/\sqrt{N_\gamma}$ slope is the
  photon-shot-noise signature.

## Localize the Phase Offset (φ₀)

The recon sinusoid is phase-shifted relative to the truth (i.e.\ the recovered
sinusoid peaks at a different $x$ than the injected one), which contributes
deterministically to the orthogonal $\delta$ even at infinite photons.  Three
single-knob tests isolate the source.  Each varies one component of the
recon pipeline while holding the rest fixed; whichever one drives $\phi_0$ to
zero is the offender.

1. **Bypass `skimage.resize`** --- interpolate from the MFT output to the pupil
   grid via HCIPy directly.  If $\phi_0 \to 0$, the resize step is shifting
   the recon by a fractional pixel.
2. **Vary `q`** (focal-plane oversampling factor) in
   `make_focal_grid` / `InstrumentConfiguration` (16 $\to$ 32 $\to$ 64).
   If $\phi_0 \propto q^{-1}$, the focal-plane sampling itself is the source.
3. **Pad-symmetrize the PSF** by averaging with its 180-deg rotation, forcing
   even-pixel centering.  If $\phi_0 \to 0$, the FFT centering convention
   (the standard fftshift / ifftshift asymmetry between even and odd grids) is
   the source.

In [ ]:
def measure_phase_offset(rec_ms_test, truth_ms_ref, pupil_mask_ref, nu0=NU0):
    """Measure the phase offset phi_0 between recon and truth sinusoids.

    Fit recon ~ a*sin(k*x + phi_0) by projecting onto sin and cos components
    of the truth frequency, then computing atan2.
    """
    # pupil_grid.x is a 1D Field; reshape to 2D then mask
    xg_2d = np.array(pupil_grid.x).reshape(pupil_mask_ref.shape)
    xg_masked = xg_2d[pupil_mask_ref]
    k = 2 * np.pi * nu0 / D
    sin_basis = np.sin(k * xg_masked)
    cos_basis = np.cos(k * xg_masked)
    # Project recon onto sin and cos at the truth frequency
    c_sin = np.sum(rec_ms_test * sin_basis) / np.sum(sin_basis**2)
    c_cos = np.sum(rec_ms_test * cos_basis) / np.sum(cos_basis**2)
    phi_0 = np.arctan2(c_cos, c_sin)  # phase offset in radians
    return np.degrees(phi_0)

# Baseline: phase offset from the standard pipeline (clean, no noise)
phi0_baseline = measure_phase_offset(rec_ms_clean, truth_ms, pupil_mask)
print(f"Baseline phase offset (clean, resize pipeline): {phi0_baseline:.1f} deg")
print("=" * 60)

### Reading the baseline $\phi_0$

$\phi_0$ is the phase shift (degrees) between the recon sinusoid and the truth sinusoid
at the same $\nu_0$, projected against the truth's sin/cos basis.

- **$|\phi_0| < 5°$:** subdominant; can be left alone for now.
- **$5° \leq |\phi_0| < 20°$:** correctable systematic.  Means the recon is producing the
  right grating at the wrong phase.  Worth fixing because it deterministically inflates
  $\delta$.
- **$|\phi_0| > 30°$:** severe coupling.  Either a pixel-centering bug or a basin-flip
  in the iteration; the three knob tests below isolate which.
- **$|\phi_0| \to 90°$ or 180°:** sign flip in the recon, not a phase shift.  Almost
  always means the diversity polarity is wrong somewhere.

### Test 1: Bypass `skimage.resize` --- use HCIPy interpolation

Resample from the MFT focal-plane grid onto the pupil grid via
`scipy.interpolate.RegularGridInterpolator` instead of `skimage.transform.resize`.
The recon RMS and the measured $\phi_0$ are the diagnostic outputs; a clean
$\phi_0 \to 0$ at fixed recon RMS indicates that resize was the offender.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

# The raw MFT output lives on its own grid; resample to the 256x256 pupil grid
# without going through skimage.resize
raw_clean_2d = np.angle(mft_rev(psf_rec_clean, conf))
n_mft = raw_clean_2d.shape[0]

# Build coordinate axes for the MFT output (normalized to pupil extent)
# mft_rev output covers the same physical pupil; pixel coords map linearly
mft_y = np.linspace(-1, 1, n_mft)
mft_x = np.linspace(-1, 1, n_mft)

# Target: 256x256 pupil grid coordinates (also normalized to [-1, 1])
tgt_y = np.linspace(-1, 1, 256)
tgt_x = np.linspace(-1, 1, 256)
tgt_yy, tgt_xx = np.meshgrid(tgt_y, tgt_x, indexing='ij')

interp = RegularGridInterpolator((mft_y, mft_x), raw_clean_2d,
                                  method='cubic', bounds_error=False, fill_value=0.0)
recon_interp = interp(np.stack([tgt_yy.ravel(), tgt_xx.ravel()], axis=-1)).reshape(256, 256)
recon_interp *= np.array(telescope_pupil.shaped)

rec_ms_interp = recon_interp[pupil_mask] - np.median(recon_interp[pupil_mask])
phi0_no_resize = measure_phase_offset(rec_ms_interp, truth_ms, pupil_mask)
alpha_no_resize = float(np.sum(rec_ms_interp * truth_ms) / np.sum(truth_ms**2))

print(f"Test 1: Bypass skimage.resize (scipy cubic interp)")
print(f"  phi_0  = {phi0_no_resize:.1f} deg   (baseline was {phi0_baseline:.1f} deg)")
print(f"  alpha  = {alpha_no_resize:.3f}       (baseline was {alpha_clean:.3f})")
print(f"  Verdict: {'resize is the culprit' if abs(phi0_no_resize) < abs(phi0_baseline) * 0.3 else 'resize is NOT the main cause'}")
print()

### Reading Test 1 (bypass `skimage.resize`)

This compares `phi_0` and `alpha` with `skimage.transform.resize` replaced by
`scipy.interpolate.RegularGridInterpolator` (cubic).  The forward model and the FDPR
iteration are unchanged; only the *final downsampling step* from the MFT focal grid back
to the pupil grid differs.

- **`phi_0` drops to near zero, `alpha` unchanged:** resize was bin-snapping the recon
  by a fractional pixel.  Use scipy interpolation in production.
- **`phi_0` unchanged, `alpha` unchanged** (the case here, both at 6.6°): the resize step
  is *not* the source of the offset.  The offset is upstream in the MFT or in the FFT
  centering.
- **`alpha` changes by more than a few percent:** the resize step was distorting the
  amplitude itself, not just the phase.  Cubic interp gives a different power normalization
  than linear; check that one of the two is closing the energy budget.

### Test 2: Vary `q` (focal-plane oversampling factor)

Rebuild the focal grid at three values of the oversampling factor $q \in \{16, 32, 64\}$
and rerun the retrieval.  A $1/q$-scaling of $\phi_0$ at fixed truth/diversity
operates as the signature: the focal-plane sampling cell width *is* the
quantization that produces the offset, and increasing $q$ makes the offset
proportionally smaller.

In [ ]:
q_values = [16, 32, 64]
q_results = []

for q_test in q_values:
    # Rebuild focal grid and propagator at this q
    fg_q = make_focal_grid(q=q_test,
                           num_airy=seal_parameters['Num_airycircles'],
                           pupil_diameter=D, focal_length=f_len,
                           reference_wavelength=wavelength)
    prop_q = FraunhoferPropagator(pupil_grid, fg_q, f_len)

    # Recompute image_dx for this q
    # image_dx = wavelength * f / (N_pix * pupil_dx), but easier to read from grid
    dx_q = float(fg_q.delta[0]) * 1e6  # m -> um

    # New InstrumentConfiguration for the MFT reverse
    conf_q = InstrumentConfiguration({
        'image_dx': dx_q,
        'efl': f_len * 1e3,
        'wavelength': wavelength * 1e6,
        'pupil_size': D * 1e3,
    })

    # Generate clean PSFs at this q
    def make_psf_q(phase_field):
        wf = Wavefront(telescope_pupil, wavelength=wavelength)
        wf.electric_field = telescope_pupil * np.exp(1j * phase_field)
        return np.array(prop_q(wf).intensity.shaped)

    pf_q = make_psf_q(phi_ab)
    pp_q = make_psf_q(phi_ab + phi_df)
    pm_q = make_psf_q(phi_ab - phi_df)

    # image_dx for FDPR (microns)
    dz_um_q = DZ_MM * 1e3

    mp_q = FocusDiversePhaseRetrieval(
        [pf_q, pp_q, pm_q],
        wavelength * 1e6,
        [dx_q, dx_q],
        [dz_um_q, -dz_um_q])
    for _ in range(N_ITERS):
        rec_q = mp_q.step()

    raw_q = np.angle(mft_rev(rec_q, conf_q))
    recon_q = resize(raw_q, (256, 256), preserve_range=True) * telescope_pupil.shaped
    r_ms_q = recon_q[pupil_mask] - np.median(recon_q[pupil_mask])

    a_q = float(np.sum(r_ms_q * truth_ms) / np.sum(truth_ms**2))
    phi0_q = measure_phase_offset(r_ms_q, truth_ms, pupil_mask)

    q_results.append((q_test, phi0_q, a_q))
    print(f"q={q_test:3d}  phi_0={phi0_q:+.1f} deg  alpha={a_q:.3f}  "
          f"image_dx={dx_q:.4f} um  PSF shape={pf_q.shape}")

print()
print("If phi_0 scales as ~1/q, focal-plane sampling is the cause.")
if len(q_results) >= 2:
    ratio = abs(q_results[0][1]) / max(abs(q_results[1][1]), 0.01)
    q_ratio = q_values[1] / q_values[0]
    print(f"  phi_0 ratio (q={q_values[0]}→{q_values[1]}): {ratio:.2f}  "
          f"(expect ~{q_ratio:.1f} if 1/q scaling)")

### Reading Test 2 (vary `q`)

If focal-plane sampling is the source of the phase offset, $\phi_0$ should scale as
$1/q$ (doubling $q$ should halve the offset).  This test sweeps $q \in \{16, 32, 64\}$.

- **$\phi_0$ halves each time you double $q$:** focal-plane sampling is the cause.
  Raising $q$ in production buys you a proportionally smaller offset; the trade-off is
  PSF array size (memory + speed).
- **$\phi_0$ unchanged with $q$:** sampling is *not* the cause.
- **$\phi_0$ *grows* with $q$** (the case here, 1.3° -> 8.4° -> 12.0°): the offset is
  coupled to the PSF array shape itself, not to the sampling pitch.  This is the signature
  of an even/odd-pixel centering convention that flips between $q$ values: when the PSF
  shape is $(2q \cdot N_{\rm airy})^2$, going from $q=16$ to $q=32$ doubles the array
  dimension, and the FFT centering convention treats odd vs even differently.  The
  resulting offset is *not* a 1/q quantization but a parity effect.

This is the test that proved the phase offset has a non-obvious origin in this notebook
-- the standard explanation (focal-plane sampling) was ruled out.

### Test 3: Pad-symmetrize PSF (force even-pixel centering)

Replace each input PSF with $\tfrac{1}{2}\bigl(\text{PSF} + \text{rot180}(\text{PSF})\bigr)$.
The rotation removes any odd-symmetric component of the PSF without changing the
encoded aberration, so $\phi_0 \to 0$ under this transformation indicates that
the FFT centering convention is producing the offset.  The cost is a small
power loss at the bright pixel; the recon RMS will rise correspondingly if
this is the offender being patched rather than the root cause being fixed.

In [ ]:
def pad_symmetrize(psf):
    """Force even-pixel centering by averaging PSF with its 180-deg rotation."""
    return 0.5 * (psf + psf[::-1, ::-1])

psf_f_sym = pad_symmetrize(psf_f_clean)
psf_p_sym = pad_symmetrize(psf_p_clean)
psf_m_sym = pad_symmetrize(psf_m_clean)

mp_sym = FocusDiversePhaseRetrieval(
    [psf_f_sym, psf_p_sym, psf_m_sym],
    WAVELENGTH_UM,
    [IMAGE_DX_UM, IMAGE_DX_UM],
    [DZ_UM, -DZ_UM])
for _ in range(N_ITERS):
    rec_sym = mp_sym.step()

raw_sym   = np.angle(mft_rev(rec_sym, conf))
recon_sym = resize(raw_sym, (256, 256), preserve_range=True) * telescope_pupil.shaped
r_ms_sym  = recon_sym[pupil_mask] - np.median(recon_sym[pupil_mask])

a_sym   = float(np.sum(r_ms_sym * truth_ms) / np.sum(truth_ms**2))
phi0_sym = measure_phase_offset(r_ms_sym, truth_ms, pupil_mask)

print(f"Test 3: Pad-symmetrize PSF")
print(f"  phi_0  = {phi0_sym:.1f} deg   (baseline was {phi0_baseline:.1f} deg)")
print(f"  alpha  = {a_sym:.3f}       (baseline was {alpha_clean:.3f})")
print(f"  Verdict: {'FFT centering is the culprit' if abs(phi0_sym) < abs(phi0_baseline) * 0.3 else 'FFT centering is NOT the main cause'}")
print()

# --- Summary table ---
print("=" * 60)
print("Phase offset localization summary")
print("=" * 60)
print(f"{'Test':<35s}  {'phi_0 (deg)':>10s}  {'alpha':>6s}")
print(f"{'-'*35}  {'-'*10}  {'-'*6}")
print(f"{'Baseline (clean + resize)':<35s}  {phi0_baseline:>+10.1f}  {alpha_clean:>6.3f}")
print(f"{'Test 1: Bypass resize (cubic)':<35s}  {phi0_no_resize:>+10.1f}  {alpha_no_resize:>6.3f}")
for q_test, phi0_q, a_q in q_results:
    print(f"{'Test 2: q=' + str(q_test):<35s}  {phi0_q:>+10.1f}  {a_q:>6.3f}")
print(f"{'Test 3: Pad-symmetrize PSF':<35s}  {phi0_sym:>+10.1f}  {a_sym:>6.3f}")

### Reading Test 3 (pad-symmetrize PSF)

This averages each input PSF with its 180° rotation, forcing even-pixel centering by
construction.  The forward model is unchanged; only the data being passed into FDPR
have had any odd-symmetric component removed.

- **$\phi_0$ drops to near zero, `alpha` roughly preserved:** FFT centering was the
  cause.  Pad-symmetrize in production, or rewrite the propagation to use a centered
  even grid throughout.
- **$\phi_0$ drops but `alpha` also drops by a few percent:** symmetrization is removing
  real information.  The phase offset *is* coming from FFT centering, but the cure costs
  amplitude.  Worth doing only if $\phi_0$ is dominating the residual.
- **$\phi_0$ blows up and `alpha` collapses or inverts** (the case here, 117.5° and
  $\alpha = -0.054$): the PSF's odd-symmetric component is *carrying the sign of the
  aberration*, and removing it destroys the recon.  This is unique to defocus diversity:
  the $+\Delta z$ and $-\Delta z$ PSFs are mirror images of each other in the
  small-signal limit, and the *difference* between them is what encodes the unknown
  phase.  Pad-symmetrize kills that difference.

Net result of Tests 1-3: **the phase offset survives all three single-knob tests.**
Its origin is some coupling between FFT centering, PSF asymmetry, and MFT-reverse
grid choice that is not isolable by any one of these knobs alone.  Resolving it requires
rebuilding the recon-side propagation on a fully-centered grid from the pupil out,
which is more work than this notebook scopes.

## Confirm δ Has No Random Component

Run the *clean* recipe (noiseless PSFs) ten times with different seeds for the
FDPR phase initialization.  The forward model is deterministic in this
configuration, so any seed-dependent variation in $\delta_{\rm rms}$ is
iteration tie-breaking rather than physics.

- $\delta_{\rm rms}$ constant to $<$ 0.5 nm across seeds $\Rightarrow$ $\delta$
  is fully geometric (MFT + resize + edge), and the photon-count sweep above
  measures a *real* pipeline floor.
- $\delta_{\rm rms}$ varies by 1--2 nm across seeds $\Rightarrow$ the amplitude-
  projection iteration is hopping between near-degenerate basins even without
  measurement noise, and the residual at any operating point carries a
  seed-uncertainty component that has to be marginalized over (or eliminated
  by moving to the Poisson-MLE formulation of \citet{Paxman1992}, as
  discussed in Section~\ref{sec:discussion} of the paper).

In [ ]:
seed_results = []

for seed in range(10):
    np.random.seed(seed)  # only affects the random phase_guess inside FDPR __init__

    mp_seed = FocusDiversePhaseRetrieval(
        [psf_f_clean, psf_p_clean, psf_m_clean],
        WAVELENGTH_UM,
        [IMAGE_DX_UM, IMAGE_DX_UM],
        [DZ_UM, -DZ_UM])
    for _ in range(N_ITERS):
        rec_seed = mp_seed.step()

    raw_seed   = np.angle(mft_rev(rec_seed, conf))
    recon_seed = resize(raw_seed, (256, 256), preserve_range=True) * telescope_pupil.shaped
    r_ms_seed  = recon_seed[pupil_mask] - np.median(recon_seed[pupil_mask])

    a_seed = float(np.sum(r_ms_seed * truth_ms) / np.sum(truth_ms**2))
    orth_seed = r_ms_seed - a_seed * truth_ms
    d_seed = rms_nm(orth_seed)
    resid_seed = rms_nm(truth_ms - r_ms_seed)
    phi0_seed = measure_phase_offset(r_ms_seed, truth_ms, pupil_mask)

    seed_results.append((seed, a_seed, d_seed, resid_seed, phi0_seed))

seed_results = np.array(seed_results)

print(f"{'seed':>4s}  {'alpha':>7s}  {'delta_rms':>9s}  {'resid_rms':>9s}  {'phi_0':>7s}")
print(f"{'----':>4s}  {'-------':>7s}  {'---------':>9s}  {'---------':>9s}  {'-------':>7s}")
for row in seed_results:
    print(f"{int(row[0]):4d}  {row[1]:7.4f}  {row[2]:9.2f}  {row[3]:9.2f}  {row[4]:+7.1f}")

print()
d_std = np.std(seed_results[:, 2])
d_mean = np.mean(seed_results[:, 2])
a_std = np.std(seed_results[:, 1])
print(f"delta_rms: mean={d_mean:.2f} nm, std={d_std:.2f} nm")
print(f"alpha:     mean={np.mean(seed_results[:, 1]):.4f}, std={a_std:.4f}")
print(f"phi_0:     mean={np.mean(seed_results[:, 4]):.1f} deg, std={np.std(seed_results[:, 4]):.1f} deg")
print()
if d_std < 0.5:
    print("VERDICT: delta is fully geometric (std < 0.5 nm). "
          "The random initial guess washes out by convergence.")
elif d_std < 2.0:
    print("VERDICT: delta has a small random component (0.5-2 nm). "
          "Iteration tie-breaking introduces some seed dependence.")
else:
    print("VERDICT: delta has a significant random component (>2 nm). "
          "The algorithm is landing in different local basins.")

### Reading the multi-seed reproducibility test

Ten different random-phase initializations on *noiseless* PSFs.  Each statistic tells
you something different about the algorithm:

- **`alpha` std $< 0.01$** (the case here, $\sigma_\alpha = 0.008$): the amplitude bias
  is essentially deterministic.  Whatever ceiling is in place doesn't depend on the
  initialization.
- **`alpha` std $> 0.02$:** the iteration is genuinely seed-dependent.  Production
  sweeps must average over at least 5 seeds, ideally 10.
- **`delta_rms` std $< 0.5$ nm:** $\delta$ is fully geometric.  No multi-trial averaging
  needed for that component.
- **`delta_rms` std $\sim 1$-2 nm** (the case here, $\sigma_\delta = 1.2$ nm): there is
  a small but non-negligible seed-uncertainty floor.  The amplitude-projection loop has
  multiple near-degenerate fixed points and the initialization picks among them.  Five
  trials/cell is enough to characterize this; one trial/cell will be slightly biased.
- **`delta_rms` std $> 5$ nm:** the cost landscape has genuinely-different basins.
  Either run more trials, or move to a deterministic (MLE) initialization.
- **`phi_0` std $> 10°$:** the phase offset is itself a seed-dependent artifact rather
  than a deterministic systematic.  Reduces the credibility of any single-trial $\phi_0$
  measurement; you need to average across seeds.

**Verdict for this notebook**: the orthogonal $\delta$ has a small (~1 nm) random
component that varies trial-to-trial; the rest of $\delta$ is deterministic pipeline
floor.